# Quant Skills — Lesson 5: Backtesting Your Setup

**Goal:** test your own strategy — **ADX trend filter + golden pocket entry** — on ten years of real
history, and learn the traps that make most backtests lie.

**How to run it:** **Runtime → Run all**. Takes about 30 seconds. Lines you can edit are marked `# <- EDIT`.

**What a backtest is:** you write your rules precisely enough that a computer can follow them, then
replay history and record every trade the rules would have taken. No hindsight, no "I would have skipped that one."

## Step 1 — Load the tools

In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Tools loaded.")

## Step 2 — Your rules, written down exactly

This is the daily-chart swing version of your setup. Every number here is a rule the computer follows.

In [ ]:
TICKERS       = ["SPY", "QQQ", "NVDA", "AMZN"]   # <- EDIT
YEARS         = 10      # <- EDIT: years of history

ADX_MIN       = 25      # <- EDIT: only trade when ADX is above this (trending market)
SWING_DAYS    = 20      # <- EDIT: look back this many days to find the swing high and low
MIN_SWING_PCT = 0.05    # <- EDIT: skip swings smaller than 5% of price (too tight to trade)
ENTRY_LEVEL   = 0.618   # golden pocket: price must reach this retracement...
POCKET_DEEP   = 0.650   # ...and this is the bottom of the pocket
STOP_LEVEL    = 0.786   # stop goes beyond the 78.6% level
TARGET_EXT    = 0.272   # <- EDIT: target = swing extreme + 27.2% of the swing (the -0.272 extension)
MAX_HOLD_DAYS = 20      # <- EDIT: time stop — exit after this many days if nothing else hits

RISK_PER_TRADE = 0.01   # 1% of the account per trade
COST_PCT       = 0.0005 # 0.05% per side for commissions + slippage — backtests without costs lie

print("Rules set.")

**The setup in plain English (long side — shorts are the mirror image):**
1. ADX above 25 and **+DI above −DI** → the market is trending up.
2. Find the swing: highest high and lowest low of the last 20 days.
3. Price pulls back into the **golden pocket** (61.8%–65% retracement) and the day **closes back above 61.8%** → buy at the close.
4. **Stop** just beyond the 78.6% level. **Target** at the −0.272 extension above the swing high.
5. If neither hits within 20 days, exit at the close.

## Step 3 — Download prices

In [ ]:
data = {}
for t in TICKERS:
    df = yf.download(t, period=f"{YEARS}y", interval="1d", auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    data[t] = df[["Open", "High", "Low", "Close"]].dropna()
    print(f"{t}: {len(data[t])} days, {data[t].index[0].date()} to {data[t].index[-1].date()}")

## Step 4 — Build ADX / DMI and the swing levels
Same Wilder formulas thinkorswim uses.

In [ ]:
def add_indicators(df, n=14, swing=SWING_DAYS):
    df = df.copy()
    h, l, c = df["High"], df["Low"], df["Close"]
    prev_c, prev_h, prev_l = c.shift(1), h.shift(1), l.shift(1)
    tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    up, down = h - prev_h, prev_l - l
    plus_dm = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)
    rma = lambda s: pd.Series(s, index=df.index).ewm(alpha=1/n, adjust=False).mean()
    atr = rma(tr)
    df["+DI"] = 100 * rma(plus_dm) / atr
    df["-DI"] = 100 * rma(minus_dm) / atr
    dx = 100 * (df["+DI"] - df["-DI"]).abs() / (df["+DI"] + df["-DI"])
    df["ADX"] = rma(dx)
    # Swing uses only PAST days (shift 1) — no peeking at today's own high/low
    df["SwingHigh"] = h.shift(1).rolling(swing).max()
    df["SwingLow"]  = l.shift(1).rolling(swing).min()
    return df.dropna()

ind = {t: add_indicators(df) for t, df in data.items()}
print("Indicators built. Latest ADX readings:")
for t, df in ind.items():
    last = df.iloc[-1]
    trend = "up" if last["+DI"] > last["-DI"] else "down"
    print(f"   {t}: ADX {last['ADX']:.1f}  ·  {trend}-trend  (+DI {last['+DI']:.1f}, -DI {last['-DI']:.1f})")

## Step 5 — Replay history, trade by trade

In [ ]:
def backtest(df, ticker, adx_min=ADX_MIN, min_swing=MIN_SWING_PCT, target_ext=TARGET_EXT):
    trades, i, idx = [], 0, df.index
    o, h, l, c = df["Open"].values, df["High"].values, df["Low"].values, df["Close"].values
    adx, pdi, mdi = df["ADX"].values, df["+DI"].values, df["-DI"].values
    sh, sl = df["SwingHigh"].values, df["SwingLow"].values
    while i < len(df) - 1:
        rng = sh[i] - sl[i]
        side = None
        if adx[i] > adx_min and rng / c[i] >= min_swing:
            if pdi[i] > mdi[i]:
                entry_lvl, deep = sh[i] - ENTRY_LEVEL * rng, sh[i] - POCKET_DEEP * rng
                if l[i] <= entry_lvl and l[i] >= sh[i] - STOP_LEVEL * rng and c[i] > entry_lvl:
                    side, stop, target = "long", sh[i] - STOP_LEVEL * rng, sh[i] + target_ext * rng
            elif mdi[i] > pdi[i]:
                entry_lvl = sl[i] + ENTRY_LEVEL * rng
                if h[i] >= entry_lvl and h[i] <= sl[i] + STOP_LEVEL * rng and c[i] < entry_lvl:
                    side, stop, target = "short", sl[i] + STOP_LEVEL * rng, sl[i] - target_ext * rng
        if side is None:
            i += 1; continue

        entry = c[i]
        risk = abs(entry - stop)
        if risk <= 0:
            i += 1; continue
        exit_px, j, reason = None, i + 1, None
        while j < len(df) and j <= i + MAX_HOLD_DAYS:
            if side == "long":
                if o[j] <= stop:   exit_px, reason = o[j], "stop (gap)"
                elif l[j] <= stop: exit_px, reason = stop, "stop"
                elif o[j] >= target: exit_px, reason = o[j], "target (gap)"
                elif h[j] >= target: exit_px, reason = target, "target"
            else:
                if o[j] >= stop:   exit_px, reason = o[j], "stop (gap)"
                elif h[j] >= stop: exit_px, reason = stop, "stop"
                elif o[j] <= target: exit_px, reason = o[j], "target (gap)"
                elif l[j] <= target: exit_px, reason = target, "target"
            if exit_px is not None: break
            j += 1
        if exit_px is None:
            j = min(j, len(df) - 1); exit_px, reason = c[j], "time stop"
        move = (exit_px - entry) if side == "long" else (entry - exit_px)
        r_mult = move / risk - (2 * COST_PCT * entry) / risk
        trades.append({"ticker": ticker, "side": side, "entry_date": idx[i], "exit_date": idx[j],
                       "entry": entry, "exit": exit_px, "R": r_mult, "reason": reason,
                       "days": j - i})
        i = j + 1
    return trades

all_trades = pd.DataFrame([t for tk, df in ind.items() for t in backtest(df, tk)])
print(f"{len(all_trades)} trades found across {len(TICKERS)} tickers.")
print(all_trades.tail(8)[["ticker", "side", "entry_date", "exit_date", "R", "reason"]]
      .assign(R=lambda d: d["R"].round(2)).to_string(index=False))

## Step 6 — The scorecard

In [ ]:
def scorecard(tr, label="All trades"):
    if len(tr) == 0:
        print(f"{label}: no trades"); return
    wins, losses = tr[tr["R"] > 0], tr[tr["R"] <= 0]
    pf = wins["R"].sum() / abs(losses["R"].sum()) if len(losses) else float("inf")
    print(f"{label}")
    print(f"   Trades:         {len(tr)}   ({len(tr)/YEARS:.0f} per year)")
    print(f"   Win rate:       {len(wins)/len(tr):.1%}")
    print(f"   Average win:    {wins['R'].mean():+.2f}R" if len(wins) else "   Average win: n/a")
    print(f"   Average loss:   {losses['R'].mean():+.2f}R" if len(losses) else "   Average loss: n/a")
    print(f"   Expectancy:     {tr['R'].mean():+.3f}R per trade   <- the number that matters")
    print(f"   Profit factor:  {pf:.2f}   (above 1.0 = profitable, above 1.5 = good)")
    print()

scorecard(all_trades)
for t in TICKERS:
    scorecard(all_trades[all_trades["ticker"] == t], t)
print("How trades ended:")
print(all_trades["reason"].value_counts().to_string())

## Step 7 — The equity curve at 1% risk per trade

Trades applied in the order they closed, risking 1% of the current balance on each. (Simplified:
it assumes you could take every signal, even when they overlap across tickers.)

In [ ]:
START = 50_000
tr = all_trades.sort_values("exit_date").reset_index(drop=True)
equity = START * np.cumprod(1 + RISK_PER_TRADE * tr["R"].values)
peak = np.maximum.accumulate(np.concatenate([[START], equity]))[1:]
dd = (equity - peak) / peak
years = max((tr["exit_date"].iloc[-1] - tr["entry_date"].iloc[0]).days / 365.25, 0.1)
cagr = (equity[-1] / START) ** (1 / years) - 1

plt.figure(figsize=(10, 4.5))
plt.plot(tr["exit_date"], equity, linewidth=1.6)
plt.axhline(START, color="black", linewidth=0.8)
plt.title("Your setup at 1% risk per trade"); plt.ylabel("Account ($)"); plt.grid(alpha=0.3); plt.show()

print(f"Start ${START:,.0f}  →  end ${equity[-1]:,.0f}   over {years:.1f} years")
print(f"Annual growth (CAGR): {cagr:.1%}")
print(f"Worst drawdown:       {dd.min():.1%}")
spy = data[TICKERS[0]]["Close"]
bh = (spy.iloc[-1] / spy.iloc[0]) ** (1 / (len(spy) / 252)) - 1
print(f"For comparison, just holding {TICKERS[0]}: {bh:.1%} a year")

## Step 8 — The trap: overfitting

The easiest way to fool yourself is to **tweak the rules until the backtest looks great**. The rules then fit
the *past* — noise included — and fall apart on new data.

The honest test: split history in two. **Tune the rules on the first 70%** (in-sample), then check them
on the **last 30% they've never seen** (out-of-sample). If the "best" settings collapse on new data, they were curve-fit.

In [ ]:
split_date = None
for t, df in ind.items():
    d = df.index[int(len(df) * 0.7)]
    split_date = d if split_date is None or d > split_date else split_date
print(f"In-sample: before {split_date.date()}  ·  Out-of-sample: after")
print()

results = []
for adx_min in (20, 25, 30):
    for min_swing in (0.03, 0.05, 0.08):
        for ext in (0.0, 0.272, 0.618):
            tr_all = pd.DataFrame([x for tk, df in ind.items()
                                   for x in backtest(df, tk, adx_min, min_swing, ext)])
            if tr_all.empty: continue
            ins = tr_all[tr_all["entry_date"] < split_date]
            oos = tr_all[tr_all["entry_date"] >= split_date]
            if len(ins) < 15 or len(oos) < 5: continue
            results.append({"ADX min": adx_min, "Min swing": min_swing, "Target ext": ext,
                            "In-sample trades": len(ins), "In-sample exp (R)": ins["R"].mean(),
                            "Out-of-sample trades": len(oos), "Out-of-sample exp (R)": oos["R"].mean()})
res = pd.DataFrame(results).sort_values("In-sample exp (R)", ascending=False)
print("Top 5 settings ranked by IN-SAMPLE expectancy — then see how they did on unseen data:")
print(res.head(5).round(3).to_string(index=False))
print()
best, default = res.iloc[0], res[(res["ADX min"] == 25) & (res["Min swing"] == 0.05) & (res["Target ext"] == 0.272)]
print(f"Best-tuned settings:   in-sample {best['In-sample exp (R)']:+.3f}R  →  out-of-sample {best['Out-of-sample exp (R)']:+.3f}R")
if len(default):
    d = default.iloc[0]
    print(f"Your default settings: in-sample {d['In-sample exp (R)']:+.3f}R  →  out-of-sample {d['Out-of-sample exp (R)']:+.3f}R")
print()
print("If the tuned settings drop a lot out-of-sample, the 'improvement' was mostly fitting noise.")

## The traps that make backtests lie — and what this notebook does about them

| Trap | What it is | How this notebook guards against it |
|---|---|---|
| **Look-ahead bias** | Using information you wouldn't have had yet | Swing levels use only *past* days (`shift(1)`) |
| **Ignoring costs** | Commissions and slippage eat thin edges | 0.05% charged on every entry and exit |
| **Optimistic fills** | Assuming the best price on gap days | Gaps through a stop fill at the open, not the stop |
| **Same-day ambiguity** | Stop and target both hit in one day | Assumes the **stop** hit first |
| **Overfitting** | Tuning until history looks perfect | Step 8: in-sample vs. out-of-sample |
| **Survivorship bias** | Only testing stocks that are still winners today | ⚠ Not fixed — NVDA and AMZN were *chosen* because they did well. Results on them look better than you'd get picking stocks in advance. |

## Your exercises

1. **Step 6:** what's the expectancy across all trades? Is it positive after costs?
2. **Step 6:** which ticker does your setup work best on — and is it the one you'd have guessed?
3. **Step 8:** do the best-tuned settings hold up out-of-sample, or collapse?
4. **Step 2:** change `TARGET_EXT` to `0.0` (target = the prior swing high) and run all. Does a closer target help or hurt?

**Next lesson:** market making — upgrading your engine into a portfolio project.